# A union B: AMN growth and MINN flux transfer

One frozen A union B FluxTransformer is shared across independently trained front MLPs. AMN uses a medium-to-growth MLP. MINN trains two independent omics-to-context MLPs: measured and predicted glucose/oxygen context. Each outer fold starts a fresh MLP. Both MINN variants use observed glucose/oxygen bounds in downstream pFBA.

The notebook contains no generic tabular-model benchmark. Run from the repository root using its Python environment. Supply the actual union checkpoint and generation provenance below. Full nested MINN evaluation is expensive (two modes ? 29 outer folds ? 50 trials ? five inner folds). Outputs remain empty until you execute it.

Implementation helpers: `iml1515_ab_evaluation.py`. Protocol: `docs/working_notes/iML1515_AB_union_notebook_plan.md`. Historical A/B/C scores require matched reruns before comparison.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata
import subprocess

import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

import iml1515_ab_evaluation as ev

ROOT = Path.cwd()
if not (ROOT / "flux_transformer.py").is_file():
    raise RuntimeError("Start the notebook in the metabolic-NN repository root.")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Run configuration

Set `CHECKPOINT` to the trained union `*_checkpoint.pth`, and `TRAINING_LOG` to its log. No existing C checkpoint is substituted. A temporary checkpoint without architecture/token metadata requires a verified JSON sidecar containing `config` and `data_info` in the canonical training-checkpoint format.

Simulated test files must be independently generated A-style and Tazza-B-style files with equal row counts. Enter actual commands, paths and seeds. `RUN_SIMULATED=False` permits experimental-only execution, with simulated validation explicitly marked incomplete. Optional target-file and cap-set sensitivities rerun both MINN modes under matching settings. C/broad fidelity and embedding exploration are deferred.

In [ ]:
CHECKPOINT = None  # Path("models/<actual_union_name>/<actual_union_name>_checkpoint.pth")
TRAINING_LOG = None  # Path("models/<actual_union_name>/<actual_union_name>_training.log")
METADATA_JSON = None  # Only needed if the checkpoint lacks config/data_info.
TRAINING_PROVENANCE = {"generation_command": "", "training_command": "", "seed": 42}

RUN_SIMULATED = True
RUN_AMN = True
RUN_MINN = True
TARGET_MODES = ["minn_fitted"]  # Optional: "non_fitted", "iml1515_minn_like".
CAP_MODES = ["co2_etoh_ac_cap"]  # Optional additional sensitivity: "etoh_ac_cap".
SIMULATED = {
    "A": {"path": None, "seed": 9, "rows": 50000, "command": ""},
    "B": {"path": None, "seed": 9, "rows": 50000, "command": ""},
}
SIM_BATCH_SIZE = 2
OBJECTIVE_CHECK_SAMPLES = 100  # First 100 rows/domain
# All rows get mass/bound checks.
PFBA_FRACTION = 0.999
XML_PATH = ROOT / "models/iML1515.xml"
AMN_DIR, MINN_DIR = ROOT / "AMN_data", ROOT / "MINN_data"

AMN_SETTINGS = dict(folds=10, split_seeds=[10, 11, 12], train_seed=10,
    inner_fraction=0.2, hidden=512, epochs=100, patience=15, min_delta=0.0,
    batch_size=1, delta=0.03, clip=1.0, amp=False, warmup=0,
    params={"drop_rate": 0.0, "learning_rate": 1e-3, "weight_decay": 1e-3})
MINN_SETTINGS = dict(seed=12345, inner_folds=5, trials=50, hidden=512,
    epochs=150, patience=25, min_delta=1e-5, batch_size=2, delta=1.0,
    clip=1.0, amp=True, warmup=10, std_penalty=0.25,
    drop_rates=[0.0, 0.05, 0.1, 0.2, 0.3, 0.35, 0.4],
    lr_range=[5e-4, 1e-2], wd_range=[1e-8, 1e-3])
# Use a new directory for every execution
# Completed folds are saved immediately.
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")


In [ ]:
required = {"CHECKPOINT": CHECKPOINT, "TRAINING_LOG": TRAINING_LOG, "XML": XML_PATH}
if METADATA_JSON is not None:
    required["METADATA_JSON"] = METADATA_JSON
if RUN_AMN:
    required.update({name: AMN_DIR / name for name in ("iML1515_EXP.csv", "EXP110.csv")})
if RUN_MINN:
    if not TARGET_MODES or len(set(TARGET_MODES)) != len(TARGET_MODES):
        raise ValueError("Specify unique MINN target modes.")
    if "co2_etoh_ac_cap" not in CAP_MODES or not set(CAP_MODES) <= {"co2_etoh_ac_cap", "etoh_ac_cap"}:
        raise ValueError("Keep the primary CO2/ethanol/acetate cap comparison enabled.")
    required.update({name: MINN_DIR / name for name in ("transcriptomics.csv", "proteomics.csv")})
    required.update({mode: MINN_DIR / ev.FLUX_FILES[mode] for mode in TARGET_MODES})
if RUN_SIMULATED:
    required.update({f"simulated_{key}": spec["path"] for key, spec in SIMULATED.items()})
missing = [name for name, path in required.items() if path is None or not Path(path).is_file()]
if missing:
    raise FileNotFoundError("Configure/provide these inputs before running: " + ", ".join(missing))
if not all(TRAINING_PROVENANCE[k] for k in ("generation_command", "training_command")):
    raise ValueError("Record actual training/generation commands in TRAINING_PROVENANCE.")
if RUN_SIMULATED and any(not s["command"] or s["seed"] == TRAINING_PROVENANCE["seed"] for s in SIMULATED.values()):
    raise ValueError("Record independent held-out generation commands/seeds.")

reservoir, input_names, output_names, architecture, training_info = ev.load_reservoir(CHECKPOINT, DEVICE, METADATA_JSON)
model_name = Path(CHECKPOINT).parent.name
run_dir = ROOT / "pics" / model_name / "AB_union_evaluation" / RUN_ID
run_dir.mkdir(parents=True, exist_ok=False)
pic_dir = run_dir / "figures"
pic_dir.mkdir()
print("Architecture:", architecture)
print("Inputs:", len(input_names), "Outputs:", len(output_names))
print("Artifacts:", run_dir)


In [ ]:
manifest = {
    "run_id": RUN_ID, "checkpoint": str(CHECKPOINT), "architecture": architecture,
    "training_info": training_info, "training_provenance": TRAINING_PROVENANCE,
    "inputs": input_names, "outputs": output_names, "device": str(DEVICE),
    "objective": ev.OBJECTIVE, "pfba_fraction": PFBA_FRACTION,
    "amn_settings": AMN_SETTINGS, "minn_settings": MINN_SETTINGS,
    "target_modes": TARGET_MODES, "cap_modes": CAP_MODES, "simulated_specs": SIMULATED,
    "objective_check_samples": OBJECTIVE_CHECK_SAMPLES,
    "versions": {p: importlib.metadata.version(p) for p in ("torch", "numpy", "pandas", "scikit-learn", "cobra", "optuna")},
    "files": {name: {"path": str(path), "sha256": ev.file_hash(path)} for name, path in required.items()},
    "code_hashes": {name: ev.file_hash(ROOT / name) for name in
        ("iml1515_ab_evaluation.py", "flux_transformer.py", "generate_ecoli_iML1515_AB_union_data.py", "ecoli_iML1515_AB_union_model_testing.ipynb")},
    "git_revision": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
    "status": "started", "deferred": ["C/intermediate fidelity", "broad D/E fidelity", "embedding exploration"],
}
ev.save_json(run_dir / "manifest.json", manifest)
# No automatic cache loading: fresh run directories prevent stale-mode/result reuse.
amn_result, minn_data, minn_results, pfba_results = None, {}, {}, {}
simulated_summary = None


## Independent simulated-data fidelity

A and B are scored separately, plus an equally weighted pooled score. Full-output inference is mandatory. The helper streams all rows, exports per-flux error/activity and per-row mass-balance/bound diagnostics, and checks objective agreement on the configured subset. A's CO2/acetate uptake semantics and B's secretion-cap semantics remain distinct. The generation commands are part of the recorded provenance.

In [ ]:
if RUN_SIMULATED:
    for spec in SIMULATED.values():
        training_path = Path(str(training_info.get("dataset", "")))
        if Path(spec["path"]).resolve() == training_path.resolve():
            raise ValueError("Simulated evaluation path points at the training dataset.")
    simulated_summary = ev.simulated_fidelity(reservoir, input_names, output_names,
        SIMULATED, XML_PATH, run_dir / "simulated", batch_size=SIM_BATCH_SIZE,
        objective_samples=OBJECTIVE_CHECK_SAMPLES, fraction=PFBA_FRACTION)
    display(simulated_summary)
else:
    print("Simulated fidelity skipped; full evaluation remains incomplete.")


## AMN: prepare A-regime media

The 110 media use base inputs of 10, fixed glycerol/amino acids of 2.2, and absent glucose/ethanol/cobalamin inputs of zero. A masked sigmoid prior predicts the ten carbon rates and oxygen. Carbon rates range from 0 to 2.2; oxygen from 0 to 10. These ranges retain the existing prior parameterization, including values below the simulation's sampling minima. Experimental SD is joined by medium composition and checked against measured growth.

In [ ]:
if RUN_AMN:
    amn_data = ev.load_amn(AMN_DIR)
    display(pd.DataFrame({"input": list(amn_data["fixed"]), "fixed_value": list(amn_data["fixed"].values())}))
    print("AMN features:", amn_data["features"])
    print("AMN shape:", amn_data["X"].shape)
    ev.save_json(run_dir / "amn_schema.json", {"features": amn_data["features"],
        "targets": amn_data["targets"], "sample_ids": amn_data["ids"].tolist(), "fixed": amn_data["fixed"]})


### Train the AMN MLP under repeated outer CV

Ten folds, stratified by carbon count, repeated with seeds 10/11/12. An inner training-only split selects the epoch count. A fresh MLP is refit on the full outer training set for that count. The outer fold is scored once. This differs from the legacy outer-fold early-stopping protocol, so historical scores are not matched comparisons.

In [ ]:
if RUN_AMN:
    amn_result = ev.run_amn(reservoir, output_names, amn_data, AMN_SETTINGS, run_dir / "amn")
    display(amn_result["summary"])
    display(amn_result["summary"].drop(columns="repeat").agg(["mean", lambda x: x.std(ddof=0)]))


### AMN growth errors and uncertainty

The main metrics are pooled within each repeat, then summarized across repeats. The averaged-repeat prediction plot is a separate diagnostic. Horizontal bars are measured SD; vertical bars are variability across split seeds, not calibrated predictive intervals.

In [ ]:
if amn_result is not None:
    oof = amn_result["oof"]
    means = oof.groupby("row").agg(truth=("truth", "first"), prediction=("prediction", "mean"), prediction_std=("prediction", "std"))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.errorbar(means.truth, means.prediction, xerr=amn_data["growth_std"][means.index],
        yerr=means.prediction_std.fillna(0), fmt="o", alpha=0.65, markersize=4)
    lo, hi = min(means.truth.min(), means.prediction.min()), max(means.truth.max(), means.prediction.max())
    ax.plot([lo, hi], [lo, hi], "k--")
    ax.set(xlabel="Measured growth (h^-1)", ylabel="OOF growth (h^-1)", title=f"{model_name}: AMN")
    fig.tight_layout()
    fig.savefig(pic_dir / f"{model_name}_amn_growth.png", dpi=180)
    plt.show()
    carbon_metrics = pd.DataFrame([{"repeat": repeat, "carbon_count": count, **ev.metrics(part.truth, part.prediction)}
        for (repeat, count), part in oof.groupby(["repeat", "carbon_count"])])
    carbon_metrics.to_csv(run_dir / "amn/carbon_count_metrics.csv", index=False)
    display(carbon_metrics)
    means.to_csv(run_dir / "amn/mean_predictions.csv")


## MINN: prepare data and B-regime context

The primary fitted target file supplies both flux targets and observed glucose/oxygen feature values, preserving the maintained Table 4 workflow. The loader checks experiment IDs across all three files. The front MLP receives 141 features; 42 signed non-context targets train its latent controls. All 47 source fluxes are retained for downstream pFBA evaluation. Common base inputs and cobalamin are 50; A-only inputs are zero.

Scaling is fitted inside each training fold. Observed glucose/oxygen are passed separately in physical units, so scaling cannot change copied reservoir bounds. Neither mode trains against exact context targets.

In [ ]:
if RUN_MINN:
    for target_mode in TARGET_MODES:
        data = ev.load_minn(MINN_DIR, output_names, target_mode)
        minn_data[target_mode] = data
        print(target_mode, "features/targets:", data["X"].shape, data["y"].shape)
        mapping = pd.DataFrame(data["mapping"], columns=["source", "token", "sign"])
        display(mapping)
        mapping.to_csv(run_dir / f"{target_mode}_mapping.csv", index=False)
        ev.save_json(run_dir / f"{target_mode}_schema.json", {"features": data["features"],
            "targets": data["targets"], "samples": data["ids"].tolist(), "fixed": data["fixed"]})


### Train MINN MLP: measured glucose/oxygen context

Outer leave-one-out CV; inner five-fold Optuna search only on the 28 training conditions. The winning trial's median inner best epoch sets the full outer-training refit length. There is no global HPO or outer-test early stopping. This MLP predicts CO2, ethanol and acetate; observed glucose/O2 are copied into reservoir inputs.

In [ ]:
if RUN_MINN:
    for target_mode, data in minn_data.items():
        minn_results[(target_mode, "measured")] = ev.run_minn(reservoir, output_names, data,
            "measured", MINN_SETTINGS, run_dir / "minn" / target_mode / "measured")


### Train a separate MINN MLP: predicted glucose/oxygen context

This mode independently predicts all five latent context values. It receives the same feature matrix and tuning budget as the measured mode. Observed glucose/O2 remain available as features and remain the downstream pFBA uptake bounds. Both modes are retained; the notebook does not select a winner using outer-test scores.

In [ ]:
if RUN_MINN:
    for target_mode, data in minn_data.items():
        minn_results[(target_mode, "predicted")] = ev.run_minn(reservoir, output_names, data,
            "predicted", MINN_SETTINGS, run_dir / "minn" / target_mode / "predicted")


### Direct MINN flux and context diagnostics

Regression R? and squared Pearson correlation (`Pearson_r2`) are separate columns. The latter preserves the historical Table 4-style correlation metric. Undefined constant-target R? or zero-denominator normalized error remains undefined. Measured-copy identity values are not reported as predictions.

In [ ]:
if RUN_MINN:
    direct_summary = pd.DataFrame([{"target_mode": target, "context_mode": mode, **result["pooled"]}
        for (target, mode), result in minn_results.items()])
    display(direct_summary)
    direct_summary.to_csv(run_dir / "minn_direct_summary.csv", index=False)
    for (target, mode), result in minn_results.items():
        print(target, mode)
        display(result["per_sample"].drop(columns="sample").agg(["mean", lambda x: x.std(ddof=0)]))
        display(result["per_flux"].sort_values("MAE", ascending=False).head(10))
        context_rows = []
        for j, source in enumerate(ev.CONTEXT_SOURCES):
            values = result["context"][:, j]
            context_rows.append({"source": source, "role": "copied" if mode == "measured" and j < 2 else "latent prediction",
                "min": values.min(), "max": values.max(), "mean": values.mean(),
                "observed_mean": minn_data[target]["context_truth"][:, j].mean()})
        display(pd.DataFrame(context_rows))
        pd.DataFrame(context_rows).to_csv(run_dir / "minn" / target / mode / "context_diagnostics.csv", index=False)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
        for ax, metric in zip(axes, ["R2", "RMSE"]):
            ax.plot(result["per_sample"][metric].to_numpy(), "o-")
            ax.set(xlabel="LOO condition index", ylabel=metric, title=f"{target}: {mode}")
        fig.tight_layout()
        fig.savefig(pic_dir / f"{model_name}_{target}_{mode}_loo.png", dpi=180)
        plt.show()


## pFBA baseline and both reservoir-to-pFBA variants

All methods preserve the same iML1515 SBML background medium and core objective. Glucose/O2 are observed uptake lower-bound caps, not equality constraints. The reservoir's B input vector follows its pretraining base-50 contract; the downstream mechanistic baseline follows the maintained Table 4 SBML medium. The configured pFBA fraction is 0.999 for all methods. OOF CO2/ethanol/acetate predictions become nonnegative secretion upper caps. Every sample retains a solver status.

In [ ]:
if RUN_MINN:
    for target, data in minn_data.items():
        pfba_results[target] = {"baseline": ev.run_pfba(XML_PATH, data, fraction=PFBA_FRACTION)}
        display(pfba_results[target]["baseline"]["status"])


In [ ]:
if RUN_MINN:
    for target, data in minn_data.items():
        for cap_mode in CAP_MODES:
            pfba_results[target][f"measured_{cap_mode}"] = ev.run_pfba(XML_PATH, data,
                minn_results[(target, "measured")]["context"], cap_mode, PFBA_FRACTION)


In [ ]:
if RUN_MINN:
    for target, data in minn_data.items():
        for cap_mode in CAP_MODES:
            pfba_results[target][f"predicted_{cap_mode}"] = ev.run_pfba(XML_PATH, data,
                minn_results[(target, "predicted")]["context"], cap_mode, PFBA_FRACTION)


### Compare methods and diagnose binding caps

Report all successes and an explicitly matched common-success subset. Optional cap-set sensitivities are compared separately so their failures cannot change the primary three-method subset. The full 47-flux and non-context-only metric sets are exported. Negative paired MAE change means improvement over baseline. `binding_low_cap` marks a binding secretion cap below its fitted target.

In [ ]:
pfba_tables = {}
if RUN_MINN:
    for target, data in minn_data.items():
        for cap_mode in CAP_MODES:
            selected = {key: pfba_results[target][key] for key in
                ("baseline", f"measured_{cap_mode}", f"predicted_{cap_mode}")}
            table = ev.compare_pfba(selected, data, run_dir / "pfba" / target / cap_mode)
            pfba_tables[(target, cap_mode)] = table
            display(table.query("coverage == 'common_success' and targets == 'all'"))
            for method, result in selected.items():
                failed = result["status"].query("status != 'optimal'")
                if len(failed):
                    print(target, method, "failed solves:")
                    display(failed)
                if not result["caps"].empty:
                    print(target, method, "binding caps:")
                    display(result["caps"].sort_values(["binding_low_cap", "slack"], ascending=[False, True]).head(12))


## Completion and exports

AMN growth, direct MINN flux predictions and pFBA flux predictions are separate outcomes. A smaller direct-flux error need not produce better pFBA caps. Missing sections or failed solves qualify completeness; no results are copied from historical checkpoints. Each MLP fit, fold assignment, selected epoch count, context prediction and solver status is stored under this run's directory.

In [ ]:
completion = {"amn": False, "minn_both_modes": False, "pfba_all_successful": False,
              "simulated_A_B": simulated_summary is not None}
if amn_result is not None:
    completion["amn"] = len(amn_result["oof"]) == 110 * len(AMN_SETTINGS["split_seeds"])
if RUN_MINN:
    completion["minn_both_modes"] = all((target, mode) in minn_results and
        len(minn_results[(target, mode)]["prediction"]) == 29 for target in TARGET_MODES for mode in ("measured", "predicted"))
    completion["pfba_all_successful"] = all(result["success"].all()
        for methods in pfba_results.values() for result in methods.values())
manifest["completion"] = completion
manifest["status"] = "complete" if all(completion.values()) else "incomplete_or_qualified"
manifest["finished_utc"] = datetime.now(timezone.utc).isoformat()
ev.save_json(run_dir / "manifest.json", manifest)
display(pd.DataFrame([completion]))
print("Run status:", manifest["status"])
print("Results:", run_dir)
print("Register actual checkpoint/provenance and matched results in docs/experiment_notes/iML1515_sampling_study_notes.md.")
